# 13 Kaggle - Sparse ALT 8-channel CNN from raw ClinVar + FASTA

Notebook Kaggle standalone cho model 8-channel sparse ALT:

```text
REF one-hot 4 channels (toàn bộ 601bp) + ALT one-hot 4 channels (chỉ tại center)
```

Không phụ thuộc `PROJECT_DIR`. Notebook tự tìm `variant_summary.txt` và FASTA GRCh38 trong `/kaggle/input`, filter ClinVar, build compact cache, train và evaluate.

So với compact (notebook 11): thay vì đưa ALT vào classifier cuối (late fusion), đưa ALT vào **đúng vị trí center** trong input tensor (early fusion). Không cần center_vector riêng, không cần SE/gating/FiLM.

## 1. Bỏ positional encoding

Notebook này mặc định **không dùng positional encoding**.

Lý do: bài toán là SNV, biến thể luôn nằm ở center của window 601bp. Với CNN 1D, thứ tự base đã nằm trong chuỗi tự nhiên. Thêm positional encoding cho model compact/sparse-ALT gốc chưa cho thấy cải thiện rõ.

Nếu muốn bật lại: set `POSITIONAL_ENCODING = "sinusoidal"`, `POSITIONAL_DIM = 8`.

## 2. Config

Các cờ mặc định không tự chạy full train. Chạy theo thứ tự:

1. Filter raw ClinVar nếu chưa có parquet.
2. Build compact REF-index nếu chưa có cache.
3. Smoke test.
4. Mini train hoặc full train.

In [49]:
from pathlib import Path
import json
import os
import random
import sys
import time
import hashlib
from collections import Counter

IS_KAGGLE = Path("/kaggle/input").exists()
INPUT_ROOT = Path("/kaggle/input") if IS_KAGGLE else Path.cwd() / "Data"
WORK_DIR = Path("/kaggle/working") if IS_KAGGLE else Path.cwd() / "kaggle_work"
PROCESSED_DIR = WORK_DIR / "processed"
MODEL_DIR = WORK_DIR / "models"
MODEL_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# Kaggle input is read-only and must come from /kaggle/input.
# Set these only if Kaggle mounted your files under unusual names.
KAGGLE_INPUT_DIR_OVERRIDE = ""        # example: "/kaggle/input/datasets/namng123g/biodemo"
VARIANT_SUMMARY_PATH_OVERRIDE = ""    # example: "/kaggle/input/.../variant_summary.txt"
FASTA_PATH_OVERRIDE = ""              # example: "/kaggle/input/.../GCF_000001405.26_GRCh38_genomic.fna"

LENGTH = 601
FLANK = LENGTH // 2
DATASET_TAG = "fixed_refseq"
LABEL_MODE = "all"                 # all = giữ Likely; definitive_only = chỉ Benign/Pathogenic, bỏ Likely
POSITIONAL_ENCODING = "none"        # none | sinusoidal | relative
POSITIONAL_DIM = 0                   # no positional channels for compact default

MODEL_VARIANT = "sparse_alt"  # compact | variant_gated_se
VARIANT_GATE_SCALE = 0.0            # conservative FiLM/SE-style channel gate strength
VARIANT_GATE_PLACEMENT = "none" # stem | stem_late | all

SPLIT_MODE = "genome_block"
GENOME_BLOCK_SIZE_BP = 1_000_000
PURGE_DISTANCE_BP = 5000
SEQUENCE_PURGE_MODE = "exact_ref"

BATCH_SIZE = 1024 if IS_KAGGLE else 256
EPOCHS = 5
HARD_NEGATIVE_EPOCHS = 2
LEARNING_RATE = 1e-3
HARD_NEGATIVE_LR = 3e-4
WEIGHT_DECAY = 1e-4
SCHEDULER = "none"
WARMUP_EPOCHS = 0
MIN_LR_RATIO = 0.05
GRAD_CLIP_NORM = 0.0
RANDOM_STATE = 42

# Kaggle speed knobs. T4/P100 thường chậm vì CPU/DataLoader/eval overhead hơn là thiếu VRAM.
NUM_WORKERS = 4 if IS_KAGGLE else 0
IMBALANCE_STRATEGY = "weighted_sampler"  # none | weighted_sampler | pos_weight | legacy_sqrt_sampler_pos_weight
PREFETCH_FACTOR = 4
PERSISTENT_WORKERS = True
PIN_MEMORY = True
USE_AMP = True
CUDNN_BENCHMARK = True
TRAIN_METRICS_MODE = "loss_only"      # loss_only | full; full sẽ tính ROC/PR-AUC train mỗi epoch và chậm hơn
EVAL_RC_TTA_EVERY_EPOCH = False       # final/test vẫn dùng RC_TTA; tắt TTA mỗi epoch để giảm gần 2x val time

RC_AUGMENT = True
RC_TTA = True
HARD_NEGATIVE_RATIO_TO_POS = 0.50
EASY_NEGATIVE_RATIO = 1.0

RUN_FILTER_RAW = False               # set True to rebuild filtered parquet even if cache exists
RUN_BUILD_COMPACT_DATASET = False    # set True to rebuild compact REF-index cache even if cache exists
RUN_DEMO_EXTRACTION = True
RUN_SMOKE_TEST = True
RUN_MINI_TRAIN = False
RUN_FULL_TRAIN = True
FORCE_OVERWRITE = True


def input_search_roots():
    roots = []
    if KAGGLE_INPUT_DIR_OVERRIDE:
        override_root = Path(KAGGLE_INPUT_DIR_OVERRIDE)
        if IS_KAGGLE and not is_under_path(override_root, Path("/kaggle/input")):
            raise ValueError(f"KAGGLE_INPUT_DIR_OVERRIDE phải nằm trong /kaggle/input: {override_root}")
        roots.append(override_root)
    roots.append(INPUT_ROOT)
    if not IS_KAGGLE:
        roots.extend([Path.cwd() / "Data", Path.cwd()])
    # De-duplicate while preserving order.
    out = []
    seen = set()
    for root in roots:
        root = Path(root)
        key = str(root.resolve()) if root.exists() else str(root)
        if key not in seen:
            seen.add(key)
            out.append(root)
    return out


def print_input_tree(max_files=120):
    print("Available input roots:")
    for root in input_search_roots():
        print("-", root, "exists=", root.exists())
    print("\nFiles visible under input roots:")
    shown = 0
    for root in input_search_roots():
        if not root.exists():
            continue
        for path in sorted(root.rglob("*")):
            if path.is_file():
                print(path)
                shown += 1
                if shown >= max_files:
                    print(f"... truncated after {max_files} files")
                    return
    if shown == 0:
        print("No files found under input roots.")


def file_candidates(root):
    root = Path(root)
    if not root.exists():
        return []
    return sorted([p for p in root.rglob("*") if p.is_file()])


def find_by_lower_name(predicate):
    matches = []
    for root in input_search_roots():
        for path in file_candidates(root):
            name = path.name.lower()
            full = str(path).lower()
            if predicate(name, full):
                matches.append(path.resolve())
    matches = sorted(set(matches))
    return matches[0] if matches else None


def is_under_path(path, root):
    path = Path(path).resolve()
    root = Path(root).resolve()
    try:
        path.relative_to(root)
        return True
    except ValueError:
        return False


def resolve_override(path_text, label):
    if not path_text:
        return None
    path = Path(path_text)
    if IS_KAGGLE and not is_under_path(path, Path("/kaggle/input")):
        raise ValueError(f"{label} phải nằm trong /kaggle/input, không đọc input từ /kaggle/working: {path}")
    if not path.exists():
        print_input_tree()
        raise FileNotFoundError(f"{label} override không tồn tại: {path}")
    return path


def discover_variant_summary():
    override = resolve_override(VARIANT_SUMMARY_PATH_OVERRIDE, "VARIANT_SUMMARY_PATH")
    if override is not None:
        return override
    found = find_by_lower_name(
        lambda name, full: (
            name in {"variant_summary.txt", "variant_summary.txt.gz"}
            or name.startswith("variant_summary")
            or "variant_summary" in full
            or ("variant" in name and "summary" in name and (name.endswith(".txt") or name.endswith(".tsv") or name.endswith(".csv") or name.endswith(".gz")))
        )
    )
    if found is not None:
        return found
    print_input_tree()
    raise FileNotFoundError(
        "Không tìm thấy variant_summary trong /kaggle/input. "
        "Set VARIANT_SUMMARY_PATH_OVERRIDE bằng path thật hiển thị ở cây file phía trên."
    )


def discover_ncbi_zip():
    return find_by_lower_name(lambda name, full: name == "ncbi_dataset.zip" or ("ncbi" in name and name.endswith(".zip")))


def discover_fasta():
    override = resolve_override(FASTA_PATH_OVERRIDE, "FASTA_PATH")
    if override is not None:
        return override

    found = find_by_lower_name(
        lambda name, full: (
            ("grch38" in name and (name.endswith(".fna") or name.endswith(".fa") or name.endswith(".fasta")))
            or name == "gcf_000001405.26_grch38_genomic.fna"
        )
    )
    if found is not None:
        return found

    zip_path = discover_ncbi_zip()
    print_input_tree()
    msg = "Không tìm thấy FASTA GRCh38 đã giải nén trong /kaggle/input."
    if zip_path is not None:
        msg += " Có thấy ncbi_dataset.zip, nhưng notebook không giải nén dataset vào /kaggle/working; hãy dùng file FASTA đã được Kaggle extract trong /kaggle/input hoặc set FASTA_PATH_OVERRIDE tới path trong /kaggle/input."
    else:
        msg += " Hãy attach dataset chứa ncbi_dataset đã giải nén hoặc set FASTA_PATH_OVERRIDE tới path trong /kaggle/input."
    raise FileNotFoundError(msg)

VARIANT_SUMMARY_PATH = discover_variant_summary()
FASTA_PATH = discover_fasta()
FILTERED_PATH = PROCESSED_DIR / "clinvar_filtered_step8_fixed_refseq.parquet"
COMPACT_METADATA_PATH = PROCESSED_DIR / f"sparse_alt_8ch_metadata_{LENGTH}_{DATASET_TAG}.parquet"
COMPACT_REF_INDEX_PATH = PROCESSED_DIR / f"X_ref_index_{LENGTH}_{DATASET_TAG}.npy"
COMPACT_Y_PATH = PROCESSED_DIR / f"y_sparse_alt_8ch_{LENGTH}_{DATASET_TAG}.npy"

POS_SUFFIX = "nopos" if POSITIONAL_DIM == 0 or POSITIONAL_ENCODING == "none" else f"{POSITIONAL_ENCODING}_pos{POSITIONAL_DIM}"
MODEL_SUFFIX = "dilated_sparse_alt"
LABEL_SUFFIX = "" if LABEL_MODE == "all" else f"_{LABEL_MODE}"
IMBALANCE_SUFFIX = "" if IMBALANCE_STRATEGY == "legacy_sqrt_sampler_pos_weight" else f"_imbalance_{IMBALANCE_STRATEGY}"
SAFE_NAME = (
    f"sparse_alt_8ch_{POS_SUFFIX}_{LENGTH}_{DATASET_TAG}_{MODEL_SUFFIX}{LABEL_SUFFIX}{IMBALANCE_SUFFIX}_"
    f"genome_block{GENOME_BLOCK_SIZE_BP}_purged{PURGE_DISTANCE_BP}_"
    f"seqpurge_{SEQUENCE_PURGE_MODE}"
)

paths = {
    "filtered": FILTERED_PATH,
    "compact_metadata": COMPACT_METADATA_PATH,
    "compact_ref_index": COMPACT_REF_INDEX_PATH,
    "compact_y": COMPACT_Y_PATH,
    "model": MODEL_DIR / f"clinvar_{SAFE_NAME}_pytorch.pt",
    "metrics": PROCESSED_DIR / f"{SAFE_NAME}_metrics.json",
    "history": PROCESSED_DIR / f"{SAFE_NAME}_training_history_pytorch.parquet",
    "predictions": PROCESSED_DIR / f"{SAFE_NAME}_test_predictions_pytorch.parquet",
    "thresholds": PROCESSED_DIR / f"{SAFE_NAME}_threshold_table_pytorch.parquet",
    "val_thresholds": PROCESSED_DIR / f"{SAFE_NAME}_val_threshold_table_pytorch.parquet",
    "split_indices": PROCESSED_DIR / f"{SAFE_NAME}_split_indices.npz",
}

print("IS_KAGGLE:", IS_KAGGLE)
print("INPUT_ROOT:", INPUT_ROOT)
print("WORK_DIR:", WORK_DIR)
print("SAFE_NAME:", SAFE_NAME)
print("VARIANT_SUMMARY:", VARIANT_SUMMARY_PATH, VARIANT_SUMMARY_PATH.exists())
print("FASTA:", FASTA_PATH, FASTA_PATH.exists())
for key, value in paths.items():
    print(f"{key:18s}", value, "exists=" + str(value.exists()))

IS_KAGGLE: True
INPUT_ROOT: /kaggle/input
WORK_DIR: /kaggle/working
SAFE_NAME: sparse_alt_8ch_nopos_601_fixed_refseq_dilated_sparse_alt_imbalance_weighted_sampler_genome_block1000000_purged5000_seqpurge_exact_ref
VARIANT_SUMMARY: /kaggle/input/datasets/namng123g/biodemo/variant_summary.txt True
FASTA: /kaggle/input/datasets/namng123g/biodemo/ncbi_dataset/ncbi_dataset/data/GCF_000001405.26/GCF_000001405.26_GRCh38_genomic.fna True
filtered           /kaggle/working/processed/clinvar_filtered_step8_fixed_refseq.parquet exists=True
compact_metadata   /kaggle/working/processed/sparse_alt_8ch_metadata_601_fixed_refseq.parquet exists=True
compact_ref_index  /kaggle/working/processed/X_ref_index_601_fixed_refseq.npy exists=True
compact_y          /kaggle/working/processed/y_sparse_alt_8ch_601_fixed_refseq.npy exists=True
model              /kaggle/working/models/clinvar_sparse_alt_8ch_nopos_601_fixed_refseq_dilated_sparse_alt_imbalance_weighted_sampler_genome_block1000000_purged5000_seqpurge_e

## Label mode

`LABEL_MODE` điều khiển có giữ nhãn `Likely` hay không:

- `"all"`: giữ toàn bộ các nhãn đã map được: `Benign`, `Likely benign`, `Pathogenic`, `Likely pathogenic`, và các nhãn gộp. Đây là cấu hình tương thích kết quả cũ.
- `"definitive_only"`: chỉ train/evaluate trên dòng có `ClinicalSignificance` đúng bằng `Benign` hoặc `Pathogenic`, bỏ `Likely ...` và nhãn gộp. Artifact sẽ có suffix `_definitive_only`.

Cache compact vẫn có thể chứa toàn bộ nhãn; notebook chỉ chọn subset trước khi split/train để tránh rebuild tensor.


## 3. Imports và helper sinh học

Mapping accession GRCh38 dùng đúng version RefSeq, tránh lỗi kiểu mọi NST đều map `.11`.

In [50]:
import subprocess

import numpy as np
import pandas as pd
try:
    from pyfaidx import Fasta
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "pyfaidx"])
    from pyfaidx import Fasta
import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    confusion_matrix,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import GroupShuffleSplit
from tqdm.auto import tqdm

CHROMOSOME_TO_REFSEQ = {
    "1": "NC_000001.11", "2": "NC_000002.12", "3": "NC_000003.12",
    "4": "NC_000004.12", "5": "NC_000005.10", "6": "NC_000006.12",
    "7": "NC_000007.14", "8": "NC_000008.11", "9": "NC_000009.12",
    "10": "NC_000010.11", "11": "NC_000011.10", "12": "NC_000012.12",
    "13": "NC_000013.11", "14": "NC_000014.9", "15": "NC_000015.10",
    "16": "NC_000016.10", "17": "NC_000017.11", "18": "NC_000018.10",
    "19": "NC_000019.10", "20": "NC_000020.11", "21": "NC_000021.9",
    "22": "NC_000022.11", "X": "NC_000023.11", "Y": "NC_000024.10",
    "MT": "NC_012920.1", "M": "NC_012920.1",
}
POSITIVE_LABELS = {"Pathogenic", "Likely pathogenic"}
NEGATIVE_LABELS = {"Benign", "Likely benign"}
TRUSTED_REVIEW_PATTERN = "criteria provided|reviewed by expert panel|practice guideline"
LOW_CONFIDENCE_REVIEW = "no assertion criteria provided"

BASE_TO_INDEX = np.full(256, 255, dtype=np.uint8)
for _idx, _base in enumerate(b"ACGT"):
    BASE_TO_INDEX[_base] = _idx
INDEX_TO_BASE = np.array(list("ACGT"))
COMPLEMENT_INDEX = np.array([3, 2, 1, 0], dtype=np.uint8)


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def split_clinical_significance(value):
    if pd.isna(value):
        return []
    return [part.strip() for part in str(value).split("/") if part.strip()]


def map_clean_label(value):
    labels = split_clinical_significance(value)
    label_set = set(labels)
    if not labels:
        return None
    if label_set <= POSITIVE_LABELS:
        return 1
    if label_set <= NEGATIVE_LABELS:
        return 0
    return None


def normalize_chromosome(chromosome):
    chrom = str(chromosome).strip()
    if chrom.lower().startswith("chr"):
        chrom = chrom[3:]
    if chrom in {"M", "MT", "m", "mt"}:
        return "MT"
    return chrom.upper()


def clean_base(value):
    return str(value).strip().upper()


def choose_chromosome_fasta(row, fasta_names):
    chrom = normalize_chromosome(row["Chromosome"])
    accession = str(row.get("ChromosomeAccession", "")).strip()
    candidates = [accession, str(row["Chromosome"]).strip(), chrom, f"chr{chrom}", CHROMOSOME_TO_REFSEQ.get(chrom, "")]
    for candidate in candidates:
        if candidate and candidate in fasta_names:
            return candidate
    return ""


def encode_sequence(sequence):
    encoded = BASE_TO_INDEX[np.frombuffer(sequence.encode("ascii"), dtype=np.uint8)]
    if np.any(encoded == 255):
        raise ValueError("sequence contains non-ACGT bases")
    return encoded.astype(np.uint8, copy=False)

set_seed(RANDOM_STATE)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
AMP_ENABLED = bool(USE_AMP and device.type == "cuda")
torch.backends.cudnn.benchmark = bool(CUDNN_BENCHMARK and device.type == "cuda")
try:
    torch.set_float32_matmul_precision("high")
except Exception:
    pass
print("device:", device)
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))
print("AMP_ENABLED:", AMP_ENABLED)
print("cudnn.benchmark:", torch.backends.cudnn.benchmark)

device: cuda
Tesla T4
AMP_ENABLED: True
cudnn.benchmark: True


## 4. Filter raw ClinVar

Cell này là bước xử lý dữ liệu từ `variant_summary.txt`. Nếu parquet đã tồn tại và `RUN_FILTER_RAW=False`, notebook dùng cache parquet này, không dùng tensor cũ.

In [51]:
def filter_variant_summary_to_parquet(output_path=FILTERED_PATH, chunksize=500_000):
    if output_path.exists() and not FORCE_OVERWRITE:
        raise FileExistsError(f"Filtered parquet exists: {output_path}")
    genome = Fasta(str(FASTA_PATH), as_raw=True, sequence_always_upper=True, rebuild=False)
    fasta_names = set(genome.keys())
    missing_primary = {k: v for k, v in CHROMOSOME_TO_REFSEQ.items() if v not in fasta_names}
    if missing_primary:
        raise RuntimeError(f"Missing primary RefSeq accessions in FASTA: {missing_primary}")

    usecols = [
        "Type", "GeneSymbol", "ClinicalSignificance", "Assembly", "ChromosomeAccession",
        "Chromosome", "ReviewStatus", "PositionVCF", "ReferenceAlleleVCF", "AlternateAlleleVCF",
    ]
    counters = Counter()
    kept_chunks = []
    reader = pd.read_csv(VARIANT_SUMMARY_PATH, sep="\t", usecols=usecols, chunksize=chunksize, low_memory=False)
    for chunk in tqdm(reader, desc="Filter variant_summary chunks"):
        counters["raw_rows"] += len(chunk)
        chunk = chunk.copy()
        chunk["Y"] = chunk["ClinicalSignificance"].map(map_clean_label)
        mask = chunk["Y"].notna()
        counters["labeled_rows"] += int(mask.sum())
        mask &= chunk["Assembly"].eq("GRCh38")
        counters["grch38_labeled_rows"] += int(mask.sum())
        mask &= chunk["Type"].eq("single nucleotide variant")
        counters["snv_rows"] += int(mask.sum())

        review_status = chunk["ReviewStatus"].fillna("").astype(str)
        trusted_review = review_status.str.contains(TRUSTED_REVIEW_PATTERN, case=False, regex=True)
        low_confidence_review = review_status.str.fullmatch(LOW_CONFIDENCE_REVIEW, case=False)
        mask &= trusted_review & ~low_confidence_review
        counters["trusted_rows"] += int(mask.sum())

        ref = chunk["ReferenceAlleleVCF"].map(clean_base)
        alt = chunk["AlternateAlleleVCF"].map(clean_base)
        position = pd.to_numeric(chunk["PositionVCF"], errors="coerce")
        mask &= ref.str.fullmatch("[ACGT]") & alt.str.fullmatch("[ACGT]")
        mask &= position.notna()
        counters["basic_allele_position_rows"] += int(mask.sum())

        out = chunk.loc[mask].copy()
        if out.empty:
            continue
        out["ReferenceAlleleVCF"] = ref.loc[mask].to_numpy()
        out["AlternateAlleleVCF"] = alt.loc[mask].to_numpy()
        out["PositionVCF"] = position.loc[mask].astype(np.int64).to_numpy()
        out["Y"] = out["Y"].astype(np.int8)
        out["Chromosome"] = out["Chromosome"].map(normalize_chromosome)
        out["ChromosomeFASTA"] = out.apply(choose_chromosome_fasta, axis=1, fasta_names=fasta_names)
        out = out[out["ChromosomeFASTA"].isin(fasta_names)].copy()
        counters["chromosome_fasta_rows"] += len(out)
        if out.empty:
            continue

        reference_bases = []
        matches = []
        for row in out.itertuples(index=False):
            base = str(genome[str(row.ChromosomeFASTA)][int(row.PositionVCF) - 1 : int(row.PositionVCF)]).upper()
            reference_bases.append(base)
            matches.append(base == row.ReferenceAlleleVCF)
        out["ReferenceBaseGRCh38"] = reference_bases
        out["ReferenceAlleleMatchesGRCh38"] = matches
        out = out[out["ReferenceAlleleMatchesGRCh38"]].copy()
        counters["ref_match_rows"] += len(out)
        if out.empty:
            continue

        ordered_cols = [
            "Type", "GeneSymbol", "ClinicalSignificance", "Assembly", "Chromosome", "ReviewStatus",
            "PositionVCF", "ReferenceAlleleVCF", "AlternateAlleleVCF", "Y", "ChromosomeFASTA",
            "ReferenceBaseGRCh38", "ReferenceAlleleMatchesGRCh38",
        ]
        kept_chunks.append(out[ordered_cols])

    if not kept_chunks:
        raise RuntimeError("No rows passed filtering.")
    filtered = pd.concat(kept_chunks, ignore_index=True)
    filtered.to_parquet(output_path, index=False)
    print("Counters:", dict(counters))
    print("Saved:", output_path)
    print("Shape:", filtered.shape)
    return filtered

if RUN_FILTER_RAW or not FILTERED_PATH.exists():
    filtered_df = filter_variant_summary_to_parquet()
else:
    filtered_df = pd.read_parquet(FILTERED_PATH)
    print("Loaded filtered parquet:", FILTERED_PATH, filtered_df.shape)
    print(filtered_df["Y"].value_counts().sort_index().to_dict())

Loaded filtered parquet: /kaggle/working/processed/clinvar_filtered_step8_fixed_refseq.parquet (1426477, 13)
{0: 1258003, 1: 168474}


## 5. Demo sparse ALT extraction

Demo này đọc trực tiếp FASTA và tạo REF-index + ALT base cho vài dòng đầu. Format: REF base index (601) + ALT base index (1 byte). Khi train, encode on-the-fly thành 8-channel tensor.

In [52]:
def extract_compact_rows(df, genome, flank=FLANK):
    context_length = flank * 2 + 1
    ref_indices = []
    metadata_rows = []
    y_values = []
    skip_counts = Counter()
    for row in tqdm(df.itertuples(index=False), total=len(df), desc="Extract compact REF index"):
        ref = clean_base(row.ReferenceAlleleVCF)
        alt = clean_base(row.AlternateAlleleVCF)
        if len(ref) != 1 or len(alt) != 1 or ref not in "ACGT" or alt not in "ACGT":
            skip_counts["bad_allele"] += 1
            continue
        chrom = str(row.ChromosomeFASTA)
        if chrom not in genome:
            skip_counts["chrom_missing"] += 1
            continue
        pos = int(row.PositionVCF)
        start_1based = pos - flank
        end_1based = pos + flank
        if start_1based < 1:
            skip_counts["edge_or_short"] += 1
            continue
        seq = str(genome[chrom][start_1based - 1 : end_1based]).upper()
        if len(seq) != context_length:
            skip_counts["edge_or_short"] += 1
            continue
        if seq[flank] != ref:
            skip_counts["ref_mismatch"] += 1
            continue
        try:
            encoded = encode_sequence(seq)
        except ValueError:
            skip_counts["non_acgt_context"] += 1
            continue
        record = row._asdict()
        record["ContextStart1Based"] = start_1based
        record["ContextEnd1Based"] = end_1based
        metadata_rows.append(record)
        ref_indices.append(encoded)
        y_values.append(int(row.Y))
    metadata_out = pd.DataFrame(metadata_rows)
    x_ref = np.vstack(ref_indices).astype(np.uint8) if ref_indices else np.empty((0, context_length), dtype=np.uint8)
    y_out = np.asarray(y_values, dtype=np.int8)
    return metadata_out, x_ref, y_out, dict(skip_counts)

if RUN_DEMO_EXTRACTION:
    genome_demo = Fasta(str(FASTA_PATH), as_raw=True, sequence_always_upper=True, rebuild=False)
    demo_meta, demo_x_ref, demo_y, demo_skips = extract_compact_rows(filtered_df.head(64), genome_demo)
    print("demo metadata:", demo_meta.shape)
    print("demo X_ref_index:", demo_x_ref.shape, demo_x_ref.dtype)
    print("demo y:", demo_y.shape, demo_y.dtype)
    print("demo skips:", demo_skips)
    print("first center REF index:", demo_x_ref[0, FLANK], "REF allele:", demo_meta.iloc[0]["ReferenceAlleleVCF"])
else:
    print("RUN_DEMO_EXTRACTION=False")

/usr/local/lib/python3.12/dist-packages/pyfaidx/__init__.py:655: RuntimeWarning: Index file /kaggle/input/datasets/namng123g/biodemo/ncbi_dataset/ncbi_dataset/data/GCF_000001405.26/GCF_000001405.26_GRCh38_genomic.fna.fai is older than FASTA file /kaggle/input/datasets/namng123g/biodemo/ncbi_dataset/ncbi_dataset/data/GCF_000001405.26/GCF_000001405.26_GRCh38_genomic.fna.
  warnings.warn(


Extract compact REF index:   0%|          | 0/64 [00:00<?, ?it/s]

demo metadata: (64, 15)
demo X_ref_index: (64, 601) uint8
demo y: (64,) int8
demo skips: {}
first center REF index: 1 REF allele: C


## 6. Build compact dataset cache

Cache là `X_ref_index`, shape `(N, 601)`, mỗi phần tử là base index `A=0,C=1,G=2,T=3`. ALT base lấy từ metadata `AlternateAlleleVCF`. Encode on-the-fly thành 8-channel tensor khi train.

In [53]:
def build_compact_dataset_cache():
    outputs = [COMPACT_METADATA_PATH, COMPACT_REF_INDEX_PATH, COMPACT_Y_PATH]
    if any(p.exists() for p in outputs) and not FORCE_OVERWRITE:
        existing = "\n".join(str(p) for p in outputs if p.exists())
        raise FileExistsError("Compact outputs exist. Set FORCE_OVERWRITE=True to overwrite:\n" + existing)
    genome = Fasta(str(FASTA_PATH), as_raw=True, sequence_always_upper=True, rebuild=False)
    metadata_out, x_ref, y_out, skip_counts = extract_compact_rows(filtered_df, genome)
    metadata_out.to_parquet(COMPACT_METADATA_PATH, index=False)
    np.save(COMPACT_Y_PATH, y_out)
    x_mem = np.lib.format.open_memmap(COMPACT_REF_INDEX_PATH, mode="w+", dtype=np.uint8, shape=x_ref.shape)
    x_mem[:] = x_ref[:]
    x_mem.flush()
    print("Saved metadata:", COMPACT_METADATA_PATH, metadata_out.shape)
    print("Saved X_ref_index:", COMPACT_REF_INDEX_PATH, x_ref.shape, x_ref.dtype)
    print("Saved y:", COMPACT_Y_PATH, y_out.shape, y_out.dtype)
    print("Skipped:", skip_counts)
    print("Label counts:", dict(zip(*np.unique(y_out, return_counts=True))))

if RUN_BUILD_COMPACT_DATASET or not (COMPACT_METADATA_PATH.exists() and COMPACT_REF_INDEX_PATH.exists() and COMPACT_Y_PATH.exists()):
    build_compact_dataset_cache()
else:
    print("Compact dataset cache exists; skip build.")

Compact dataset cache exists; skip build.


## 7. Load compact dataset

In [54]:
metadata = pd.read_parquet(COMPACT_METADATA_PATH)
x_ref_index = np.load(COMPACT_REF_INDEX_PATH, mmap_mode="r")
y = np.load(COMPACT_Y_PATH, mmap_mode="r")
assert x_ref_index.ndim == 2 and x_ref_index.shape[1] == LENGTH
assert len(metadata) == len(x_ref_index) == len(y)
print("metadata:", metadata.shape)
print("X_ref_index:", x_ref_index.shape, x_ref_index.dtype)
print("y:", y.shape, y.dtype)
print("labels:", dict(zip(*np.unique(y, return_counts=True))))

def select_label_indices_for_mode(meta, label_mode):
    if label_mode == "all":
        return np.arange(len(meta), dtype=np.int64)
    if label_mode == "definitive_only":
        clinical = meta["ClinicalSignificance"].fillna("").astype(str).str.strip()
        return np.flatnonzero(clinical.isin(["Benign", "Pathogenic"]).to_numpy()).astype(np.int64)
    raise ValueError(f"Unsupported LABEL_MODE: {label_mode}")

active_idx = select_label_indices_for_mode(metadata, LABEL_MODE)
metadata_for_split = metadata.iloc[active_idx].reset_index(drop=True)
y_for_split = np.asarray(y[active_idx], dtype=np.int8)
print("LABEL_MODE:", LABEL_MODE)
print("active rows:", f"{len(active_idx):,}", "/", f"{len(metadata):,}")
print("active labels:", dict(zip(*np.unique(y_for_split, return_counts=True))))


metadata: (1426427, 15)
X_ref_index: (1426427, 601) uint8
y: (1426427,) int8
labels: {np.int8(0): np.int64(1257967), np.int8(1): np.int64(168460)}
LABEL_MODE: all
active rows: 1,426,427 / 1,426,427
active labels: {np.int8(0): np.int64(1257967), np.int8(1): np.int64(168460)}


## 8. Split strict: genome block + coordinate purge + sequence purge

In [55]:
def make_groups(meta):
    groups = meta["GeneSymbol"].fillna("unknown").astype(str).to_numpy(copy=True)
    unknown = (groups == "") | (groups == "unknown") | (groups == "nan")
    row_ids = np.arange(len(groups)).astype(str)
    groups[unknown] = np.char.add("unknown_row_", row_ids[unknown])
    return groups


def make_chromosome_groups(meta):
    groups = meta["Chromosome"].fillna("unknown").astype(str).str.replace("chr", "", case=False, regex=False).to_numpy(copy=True)
    unknown = (groups == "") | (groups == "unknown") | (groups == "nan")
    row_ids = np.arange(len(groups)).astype(str)
    groups[unknown] = np.char.add("unknown_chr_row_", row_ids[unknown])
    return groups


def make_genome_block_groups(meta, chromosome_groups, block_size_bp):
    positions = pd.to_numeric(meta["PositionVCF"], errors="coerce")
    blocks = ((positions - 1) // block_size_bp).astype("Int64")
    chrom = pd.Series(chromosome_groups, index=meta.index).astype(str)
    block_groups = chrom + ":block_" + blocks.astype(str)
    invalid = positions.isna() | chrom.isin(["", "unknown", "nan"]) | blocks.isna()
    if invalid.any():
        row_ids = pd.Series(np.arange(len(meta)), index=meta.index).astype(str)
        block_groups.loc[invalid] = "unknown_block_row_" + row_ids.loc[invalid]
    return block_groups.to_numpy(dtype=str, copy=True)


def make_group_split(labels, groups, random_state):
    all_idx = np.arange(len(labels))
    gss1 = GroupShuffleSplit(n_splits=1, test_size=0.30, random_state=random_state)
    train_idx, temp_idx = next(gss1.split(all_idx, labels, groups=groups))
    gss2 = GroupShuffleSplit(n_splits=1, test_size=0.50, random_state=random_state)
    val_rel, test_rel = next(gss2.split(temp_idx, labels[temp_idx], groups=groups[temp_idx]))
    return train_idx, temp_idx[val_rel], temp_idx[test_rel]


def nearest_distance_to_reference(meta, query_idx, reference_idx):
    ref_pos_by_chr = {}
    for chrom, sub in meta.iloc[reference_idx].groupby("Chromosome", sort=False):
        positions = pd.to_numeric(sub["PositionVCF"], errors="coerce").dropna()
        ref_pos_by_chr[str(chrom)] = np.sort(positions.to_numpy(dtype=np.int64))
    query = meta.iloc[query_idx]
    distances = np.full(len(query_idx), np.inf, dtype=np.float64)
    for i, (chrom, pos) in enumerate(zip(query["Chromosome"].astype(str), pd.to_numeric(query["PositionVCF"], errors="coerce"))):
        if pd.isna(pos):
            continue
        ref_positions = ref_pos_by_chr.get(str(chrom))
        if ref_positions is None or len(ref_positions) == 0:
            continue
        pos_int = int(pos)
        insert_at = np.searchsorted(ref_positions, pos_int)
        best = np.inf
        if insert_at > 0:
            best = min(best, abs(pos_int - int(ref_positions[insert_at - 1])))
        if insert_at < len(ref_positions):
            best = min(best, abs(pos_int - int(ref_positions[insert_at])))
        distances[i] = best
    return distances


def distance_summary(distances):
    finite = distances[np.isfinite(distances)]
    if len(finite) == 0:
        return {"min": float("inf"), "p5": float("inf"), "median": float("inf"), "p95": float("inf")}
    return {"min": float(np.min(finite)), "p5": float(np.percentile(finite, 5)), "median": float(np.median(finite)), "p95": float(np.percentile(finite, 95))}


def purge_nearby_splits(meta, train_idx, val_idx, test_idx, purge_distance_bp):
    val_dist = nearest_distance_to_reference(meta, val_idx, train_idx)
    keep_val = val_dist >= purge_distance_bp
    kept_val_idx = val_idx[keep_val]
    test_ref = np.concatenate([train_idx, kept_val_idx])
    test_dist = nearest_distance_to_reference(meta, test_idx, test_ref)
    keep_test = test_dist >= purge_distance_bp
    kept_test_idx = test_idx[keep_test]
    audit = {
        "val_rows_before_purge": int(len(val_idx)),
        "val_rows_after_purge": int(len(kept_val_idx)),
        "val_rows_purged": int((~keep_val).sum()),
        "test_rows_before_purge": int(len(test_idx)),
        "test_rows_after_purge": int(len(kept_test_idx)),
        "test_rows_purged": int((~keep_test).sum()),
        "val_nearest_train_distance_bp": distance_summary(nearest_distance_to_reference(meta, kept_val_idx, train_idx)),
        "test_nearest_train_val_distance_bp": distance_summary(nearest_distance_to_reference(meta, kept_test_idx, test_ref)),
    }
    return kept_val_idx, kept_test_idx, audit


def ref_context_hash(index):
    return hashlib.blake2b(np.ascontiguousarray(x_ref_index[int(active_idx[int(index)])]).view(np.uint8), digest_size=16).digest()


def purge_sequence_duplicates(train_idx, val_idx, test_idx):
    train_hashes = {ref_context_hash(i) for i in tqdm(train_idx, desc="hash train REF contexts", leave=False)}
    keep_val = np.array([ref_context_hash(i) not in train_hashes for i in tqdm(val_idx, desc="purge val REF duplicates", leave=False)])
    kept_val_idx = val_idx[keep_val]
    val_hashes = {ref_context_hash(i) for i in tqdm(kept_val_idx, desc="hash kept val REF contexts", leave=False)}
    reference_hashes = train_hashes | val_hashes
    keep_test = np.array([ref_context_hash(i) not in reference_hashes for i in tqdm(test_idx, desc="purge test REF duplicates", leave=False)])
    kept_test_idx = test_idx[keep_test]
    audit = {
        "sequence_purge_mode": SEQUENCE_PURGE_MODE,
        "val_rows_before_sequence_purge": int(len(val_idx)),
        "val_rows_after_sequence_purge": int(len(kept_val_idx)),
        "val_rows_purged_by_sequence": int((~keep_val).sum()),
        "test_rows_before_sequence_purge": int(len(test_idx)),
        "test_rows_after_sequence_purge": int(len(kept_test_idx)),
        "test_rows_purged_by_sequence": int((~keep_test).sum()),
    }
    return kept_val_idx, kept_test_idx, audit


def label_count_dict(indices):
    labels, counts = np.unique(y_for_split[indices], return_counts=True)
    return {int(k): int(v) for k, v in zip(labels, counts)}

chromosome_groups = make_chromosome_groups(metadata_for_split)
genome_block_groups = make_genome_block_groups(metadata_for_split, chromosome_groups, GENOME_BLOCK_SIZE_BP)
train_idx, val_idx, test_idx = make_group_split(y_for_split, genome_block_groups, RANDOM_STATE)
val_idx, test_idx, coordinate_purge_audit = purge_nearby_splits(metadata_for_split, train_idx, val_idx, test_idx, PURGE_DISTANCE_BP)
val_idx, test_idx, sequence_purge_audit = purge_sequence_duplicates(train_idx, val_idx, test_idx)

train_blocks = set(genome_block_groups[train_idx])
val_blocks = set(genome_block_groups[val_idx])
test_blocks = set(genome_block_groups[test_idx])
split_audit = {
    "split_mode": SPLIT_MODE,
    "genome_block_size_bp": GENOME_BLOCK_SIZE_BP,
    "purge_distance_bp": PURGE_DISTANCE_BP,
    "sequence_purge_mode": SEQUENCE_PURGE_MODE,
    "train_rows": int(len(train_idx)),
    "val_rows": int(len(val_idx)),
    "test_rows": int(len(test_idx)),
    "train_label_counts": label_count_dict(train_idx),
    "val_label_counts": label_count_dict(val_idx),
    "test_label_counts": label_count_dict(test_idx),
    "genome_block_overlap_train_val": int(len(train_blocks & val_blocks)),
    "genome_block_overlap_train_test": int(len(train_blocks & test_blocks)),
    "genome_block_overlap_val_test": int(len(val_blocks & test_blocks)),
    "val_nearest_train_distance_bp": distance_summary(nearest_distance_to_reference(metadata_for_split, val_idx, train_idx)),
    "test_nearest_train_val_distance_bp": distance_summary(nearest_distance_to_reference(metadata_for_split, test_idx, np.concatenate([train_idx, val_idx]))),
    "coordinate_purge_audit": coordinate_purge_audit,
    "sequence_purge_audit": sequence_purge_audit,
}
print(json.dumps(split_audit, indent=2))
assert split_audit["genome_block_overlap_train_val"] == 0
assert split_audit["genome_block_overlap_train_test"] == 0
assert split_audit["genome_block_overlap_val_test"] == 0
assert split_audit["val_nearest_train_distance_bp"]["min"] >= PURGE_DISTANCE_BP
assert split_audit["test_nearest_train_val_distance_bp"]["min"] >= PURGE_DISTANCE_BP

train_idx_local = train_idx
val_idx_local = val_idx
test_idx_local = test_idx
train_idx = active_idx[train_idx_local]
val_idx = active_idx[val_idx_local]
test_idx = active_idx[test_idx_local]
print("global train/val/test rows:", len(train_idx), len(val_idx), len(test_idx))

hash train REF contexts:   0%|          | 0/1002376 [00:00<?, ?it/s]

purge val REF duplicates:   0%|          | 0/205875 [00:00<?, ?it/s]

hash kept val REF contexts:   0%|          | 0/205864 [00:00<?, ?it/s]

purge test REF duplicates:   0%|          | 0/216364 [00:00<?, ?it/s]

{
  "split_mode": "genome_block",
  "genome_block_size_bp": 1000000,
  "purge_distance_bp": 5000,
  "sequence_purge_mode": "exact_ref",
  "train_rows": 1002376,
  "val_rows": 205864,
  "test_rows": 216351,
  "train_label_counts": {
    "0": 883610,
    "1": 118766
  },
  "val_label_counts": {
    "0": 180524,
    "1": 25340
  },
  "test_label_counts": {
    "0": 192330,
    "1": 24021
  },
  "genome_block_overlap_train_val": 0,
  "genome_block_overlap_train_test": 0,
  "genome_block_overlap_val_test": 0,
  "val_nearest_train_distance_bp": {
    "min": 5001.0,
    "p5": 47660.3,
    "median": 398213.0,
    "p95": 1370029.0999999999
  },
  "test_nearest_train_val_distance_bp": {
    "min": 5023.0,
    "p5": 47718.0,
    "median": 388904.0,
    "p95": 1328722.5
  },
  "coordinate_purge_audit": {
    "val_rows_before_purge": 206990,
    "val_rows_after_purge": 205875,
    "val_rows_purged": 1115,
    "test_rows_before_purge": 217061,
    "test_rows_after_purge": 216364,
    "test_rows_purg

## 9. Positional encoding và Dataset sparse ALT 8-channel

In [56]:
def make_position_encoding(kind, length, dim):
    if dim == 0 or kind == "none":
        return np.zeros((length, 0), dtype=np.float32)
    positions = np.arange(length, dtype=np.float32)[:, None]
    if kind == "relative" and dim == 1:
        center = (length - 1) / 2.0
        return ((positions - center) / max(center, 1.0)).astype(np.float32)
    if dim % 2 != 0:
        raise ValueError("Sinusoidal positional encoding requires even POSITIONAL_DIM; use relative dim=1 for scalar position.")
    if kind == "relative":
        positions = positions - (length - 1) / 2.0
    div_term = np.exp(np.arange(0, dim, 2, dtype=np.float32) * (-np.log(10000.0) / dim))
    pe = np.zeros((length, dim), dtype=np.float32)
    pe[:, 0::2] = np.sin(positions * div_term)
    pe[:, 1::2] = np.cos(positions * div_term)
    return pe


POSITIONAL_ENCODING_ARRAY = make_position_encoding(POSITIONAL_ENCODING, LENGTH, POSITIONAL_DIM)
SEQ_CHANNELS = 8 + POSITIONAL_ENCODING_ARRAY.shape[1]
CENTER_VARIANT_DIM = 0 


BASE_TO_INDEX_MAP = {"A": 0, "C": 1, "G": 2, "T": 3}


class SparseAlt8chDataset(Dataset):
    """Dataset that encodes REF one-hot (4ch) + ALT one-hot at center only (4ch) = 8 channels on-the-fly."""
    def __init__(self, indices, positional_encoding=None, random_reverse_complement=False, reverse_complement=False):
        self.indices = np.asarray(indices, dtype=np.int64)
        self.positional_encoding = positional_encoding
        self.random_reverse_complement = random_reverse_complement
        self.reverse_complement = reverse_complement
        if random_reverse_complement and reverse_complement:
            raise ValueError("Use either random or deterministic reverse-complement, not both.")
        # Precompute ALT base index from metadata
        alt_col = metadata["AlternateAlleleVCF"].astype(str).str.strip().str.upper()
        self.alt_base_index = alt_col.map(BASE_TO_INDEX_MAP).to_numpy(dtype=np.uint8)

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, item):
        idx = int(self.indices[item])
        ref_idx = x_ref_index[idx]  # shape (LENGTH,), values 0-3

        # Build 8-channel tensor: 4 REF one-hot + 4 ALT one-hot (sparse, only at center)
        x = np.zeros((LENGTH, 8), dtype=np.float32)
        positions = np.arange(LENGTH)
        x[positions, ref_idx] = 1.0                         # REF one-hot ch 0-3
        x[FLANK, 4 + self.alt_base_index[idx]] = 1.0        # ALT one-hot ch 4-7, only at center

        if self.reverse_complement or (self.random_reverse_complement and random.random() < 0.5):
            x = x[::-1].copy()
            x[:, 0:4] = x[:, [3, 2, 1, 0]]    # complement REF channels
            x[:, 4:8] = x[:, [7, 6, 5, 4]]    # complement ALT channels

        if self.positional_encoding is not None and self.positional_encoding.shape[1] > 0:
            x = np.concatenate([x, self.positional_encoding], axis=1)

        x = np.ascontiguousarray(x.T)  # (channels, LENGTH)
        label = np.float32(y[idx])
        return torch.from_numpy(x), torch.tensor(label)


def make_loader(indices, batch_size, sampler=None, shuffle=False, random_reverse_complement=False, reverse_complement=False):
    pe = POSITIONAL_ENCODING_ARRAY if POSITIONAL_DIM > 0 and POSITIONAL_ENCODING != "none" else None
    ds = SparseAlt8chDataset(
        indices,
        positional_encoding=pe,
        random_reverse_complement=random_reverse_complement,
        reverse_complement=reverse_complement,
    )
    loader_kwargs = {"batch_size": batch_size, "shuffle": shuffle if sampler is None else False, "sampler": sampler, "pin_memory": PIN_MEMORY}
    if NUM_WORKERS > 0:
        loader_kwargs.update({"num_workers": NUM_WORKERS, "prefetch_factor": PREFETCH_FACTOR, "persistent_workers": PERSISTENT_WORKERS})
    else:
        loader_kwargs["num_workers"] = 0
    return DataLoader(ds, **loader_kwargs)


def make_weighted_sampler(indices):
    if IMBALANCE_STRATEGY not in {"weighted_sampler", "legacy_sqrt_sampler_pos_weight"}:
        return None
    labels = y[indices].astype(int)
    counts = np.bincount(labels, minlength=2)
    if IMBALANCE_STRATEGY == "weighted_sampler":
        weights_by_class = 1.0 / np.maximum(counts, 1)
    else:
        weights_by_class = 1.0 / np.sqrt(np.maximum(counts, 1))
    sample_weights = weights_by_class[labels]
    return WeightedRandomSampler(torch.as_tensor(sample_weights, dtype=torch.double), num_samples=len(sample_weights), replacement=True)


def compute_pos_weight(indices):
    labels = y[indices].astype(int)
    counts = np.bincount(labels, minlength=2)
    if IMBALANCE_STRATEGY == "pos_weight":
        return float(max(counts[0], 1) / max(counts[1], 1))
    if IMBALANCE_STRATEGY == "legacy_sqrt_sampler_pos_weight":
        return float(np.sqrt(max(counts[0], 1) / max(counts[1], 1)))
    return 1.0

print("SEQ_CHANNELS:", SEQ_CHANNELS)
print("CENTER_VARIANT_DIM:", CENTER_VARIANT_DIM)

SEQ_CHANNELS: 8
CENTER_VARIANT_DIM: 0


## 10. Model: Dilated CNN for 8-channel input

Model giống dilated CNN chuẩn (notebook 10), không cần center_vector fusion hay SE/gating/FiLM.
Khác biệt duy nhất so với full 9-channel: bỏ marker channel, ALT channels sparse.

In [57]:
class DilatedClinVarCNN(nn.Module):
    """Dilated CNN for 8-channel sparse ALT input. Same architecture as full 9-channel model."""
    def __init__(self, in_channels):
        super().__init__()
        layers = [
            nn.Conv1d(in_channels, 64, kernel_size=7, padding=3),
            nn.BatchNorm1d(64),
            nn.ReLU(),
        ]
        for dilation in [1, 2, 4, 8, 16, 32, 64]:
            layers.extend([
                nn.Conv1d(64, 64, kernel_size=7, padding=3 * dilation, dilation=dilation),
                nn.BatchNorm1d(64),
                nn.ReLU(),
                nn.Dropout(0.05),
            ])
        self.features = nn.Sequential(*layers)
        self.classifier = nn.Sequential(
            nn.Linear(64 * 3, 96),
            nn.ReLU(),
            nn.Dropout(0.25),
            nn.Linear(96, 1),
        )

    def forward(self, x):
        features = self.features(x)
        center = features[:, :, features.shape[-1] // 2]
        global_max = torch.amax(features, dim=-1)
        global_mean = torch.mean(features, dim=-1)
        pooled = torch.cat([center, global_max, global_mean], dim=1)
        return self.classifier(pooled).squeeze(1)


def make_sparse_alt_model():
    in_ch = SEQ_CHANNELS
    return DilatedClinVarCNN(in_ch)

## 11. Metrics không dùng threshold 0.5 làm chuẩn

In [58]:
def safe_auc(y_true, y_score, kind):
    if len(np.unique(y_true)) < 2:
        return float("nan")
    return float(roc_auc_score(y_true, y_score) if kind == "roc" else average_precision_score(y_true, y_score))


def metrics_from_probs(y_true, y_prob, loss):
    return {"loss": float(loss), "roc_auc": safe_auc(y_true, y_prob, "roc"), "pr_auc": safe_auc(y_true, y_prob, "pr")}


def threshold_table(y_true, y_prob):
    precision, recall, thresholds = precision_recall_curve(y_true, y_prob)
    table = pd.DataFrame({"threshold": thresholds, "precision": precision[:-1], "recall": recall[:-1]})
    denom = table["precision"] + table["recall"]
    table["f1"] = np.where(denom > 0, 2 * table["precision"] * table["recall"] / denom, 0.0)
    beta2 = 4.0
    denom_f2 = beta2 * table["precision"] + table["recall"]
    table["f2"] = np.where(denom_f2 > 0, (1.0 + beta2) * table["precision"] * table["recall"] / denom_f2, 0.0)
    return table


def metrics_at_threshold(y_true, y_prob, threshold):
    y_pred = (y_prob >= threshold).astype(int)
    return {
        "threshold": float(threshold),
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "precision_pathogenic": float(precision_score(y_true, y_pred, zero_division=0)),
        "recall_pathogenic": float(recall_score(y_true, y_pred, zero_division=0)),
        "f1_pathogenic": float(f1_score(y_true, y_pred, zero_division=0)),
        "confusion_matrix": confusion_matrix(y_true, y_pred).tolist(),
    }


def best_recall_at_precision(table, min_precision=0.80):
    candidates = table[table["precision"] >= min_precision]
    if candidates.empty:
        return None
    return candidates.sort_values(["recall", "precision"], ascending=False).iloc[0].to_dict()


def best_precision_at_recall(table, min_recall=0.90):
    candidates = table[table["recall"] >= min_recall]
    if candidates.empty:
        return None
    return candidates.sort_values(["precision", "recall"], ascending=False).iloc[0].to_dict()

## 12. Train/eval helpers

In [59]:
def make_scheduler(optimizer, total_steps):
    if SCHEDULER == "none" or total_steps <= 0:
        return None
    warmup_steps = max(0, min(int(WARMUP_EPOCHS * total_steps), total_steps - 1))
    if SCHEDULER == "cosine":
        def lr_lambda(step):
            if warmup_steps > 0 and step < warmup_steps:
                return max((step + 1) / warmup_steps, MIN_LR_RATIO)
            progress = min(1.0, max(0.0, (step - warmup_steps) / max(1, total_steps - warmup_steps)))
            cosine = 0.5 * (1.0 + np.cos(np.pi * progress))
            return MIN_LR_RATIO + (1.0 - MIN_LR_RATIO) * cosine
        return torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
    raise ValueError(f"Unsupported scheduler in notebook: {SCHEDULER}")


def run_epoch(model, loader, criterion, optimizer, label, scheduler=None, scaler=None):
    train = optimizer is not None
    model.train(train)
    collect_outputs = (not train) or TRAIN_METRICS_MODE == "full"
    loss_sum = 0.0
    row_count = 0
    probs_all, labels_all = [], []
    last_lr = float(optimizer.param_groups[0]["lr"]) if train else float("nan")
    last_grad_norm = float("nan")

    for x_batch, labels in tqdm(loader, desc=label, leave=False):
        x_batch = x_batch.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        if train:
            optimizer.zero_grad(set_to_none=True)

        with torch.set_grad_enabled(train):
            with torch.cuda.amp.autocast(enabled=AMP_ENABLED):
                logits = model(x_batch)
                loss = criterion(logits, labels)

            if train:
                if scaler is not None and AMP_ENABLED:
                    scaler.scale(loss).backward()
                    if GRAD_CLIP_NORM and GRAD_CLIP_NORM > 0:
                        scaler.unscale_(optimizer)
                        grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP_NORM)
                        last_grad_norm = float(grad_norm.detach().cpu())
                    scaler.step(optimizer)
                    scaler.update()
                else:
                    loss.backward()
                    if GRAD_CLIP_NORM and GRAD_CLIP_NORM > 0:
                        grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP_NORM)
                        last_grad_norm = float(grad_norm.detach().cpu())
                    optimizer.step()
                if scheduler is not None:
                    scheduler.step()
                last_lr = float(optimizer.param_groups[0]["lr"])

        batch_size = int(labels.shape[0])
        loss_sum += float(loss.item()) * batch_size
        row_count += batch_size
        if collect_outputs:
            probs_all.append(torch.sigmoid(logits).detach().cpu().numpy())
            labels_all.append(labels.detach().cpu().numpy())

    if collect_outputs:
        y_true = np.concatenate(labels_all).astype(int)
        y_prob = np.concatenate(probs_all)
        metrics = metrics_from_probs(y_true, y_prob, loss_sum / max(row_count, 1))
    else:
        metrics = {"loss": loss_sum / max(row_count, 1)}

    if train:
        metrics["lr"] = last_lr
        metrics["grad_norm"] = last_grad_norm
    return metrics, (np.concatenate(probs_all) if probs_all else None), (np.concatenate(labels_all).astype(int) if labels_all else None)

def run_eval_epoch(model, loader, rc_loader, criterion, label, use_rc_tta):
    metrics, probs, labels = run_epoch(model, loader, criterion, None, label)
    if not use_rc_tta:
        return metrics, probs, labels
    rc_metrics, rc_probs, rc_labels = run_epoch(model, rc_loader, criterion, None, label + " rc")
    if not np.array_equal(labels, rc_labels):
        raise ValueError("RC loader label order mismatch")
    tta_probs = (probs + rc_probs) / 2.0
    tta_loss = (metrics["loss"] + rc_metrics["loss"]) / 2.0
    return metrics_from_probs(labels, tta_probs, tta_loss), tta_probs, labels

## 13. Smoke test

In [60]:
if RUN_SMOKE_TEST:
    smoke_loader = make_loader(train_idx[:8], batch_size=8)
    x_batch, y_batch = next(iter(smoke_loader))
    print("input:", tuple(x_batch.shape))
    print("labels:", tuple(y_batch.shape))
    assert tuple(x_batch.shape) == (8, SEQ_CHANNELS, LENGTH), f"Expected (8, {SEQ_CHANNELS}, {LENGTH}), got {tuple(x_batch.shape)}"
    model = make_sparse_alt_model().to(device)
    criterion = nn.BCEWithLogitsLoss()
    logits = model(x_batch.to(device))
    loss = criterion(logits, y_batch.to(device))
    loss.backward()
    print("logits:", tuple(logits.shape), "loss:", float(loss.detach().cpu()))
    del model
    torch.cuda.empty_cache() if torch.cuda.is_available() else None
else:
    print("RUN_SMOKE_TEST=False")

input: (8, 8, 601)
labels: (8,)
logits: (8,) loss: 0.7492138147354126


## 14. Training function

Hard-negative mining lấy top-scored negatives, không dựa vào `threshold=0.5`.

In [61]:
def train_compact_model(train_indices, val_indices, test_indices, run_name, save_outputs=True, epochs=EPOCHS, hard_negative_epochs=HARD_NEGATIVE_EPOCHS, batch_size=BATCH_SIZE):
    output_paths = {
        "model": MODEL_DIR / f"clinvar_{run_name}_pytorch.pt",
        "metrics": PROCESSED_DIR / f"{run_name}_metrics.json",
        "history": PROCESSED_DIR / f"{run_name}_training_history_pytorch.parquet",
        "predictions": PROCESSED_DIR / f"{run_name}_test_predictions_pytorch.parquet",
        "thresholds": PROCESSED_DIR / f"{run_name}_threshold_table_pytorch.parquet",
        "val_thresholds": PROCESSED_DIR / f"{run_name}_val_threshold_table_pytorch.parquet",
        "split_indices": PROCESSED_DIR / f"{run_name}_split_indices.npz",
    }
    if save_outputs and any(p.exists() for p in output_paths.values()) and not FORCE_OVERWRITE:
        existing = "\n".join(str(p) for p in output_paths.values() if p.exists())
        raise FileExistsError("Output exists. Set FORCE_OVERWRITE=True to overwrite:\n" + existing)

    train_sampler = make_weighted_sampler(train_indices)
    train_loader = make_loader(
        train_indices,
        batch_size=batch_size,
        sampler=train_sampler,
        shuffle=train_sampler is None,
        random_reverse_complement=RC_AUGMENT,
    )
    train_eval_loader = make_loader(train_indices, batch_size=batch_size)
    train_eval_rc_loader = make_loader(train_indices, batch_size=batch_size, reverse_complement=True) if RC_TTA else None
    val_loader = make_loader(val_indices, batch_size=batch_size)
    val_rc_loader = make_loader(val_indices, batch_size=batch_size, reverse_complement=True) if RC_TTA else None
    test_loader = make_loader(test_indices, batch_size=batch_size)
    test_rc_loader = make_loader(test_indices, batch_size=batch_size, reverse_complement=True) if RC_TTA else None

    pos_weight_value = compute_pos_weight(train_indices)
    if pos_weight_value == 1.0:
        criterion = nn.BCEWithLogitsLoss()
    else:
        criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor(pos_weight_value, dtype=torch.float32, device=device))
    model = make_sparse_alt_model().to(device)
    scaler = torch.cuda.amp.GradScaler(enabled=AMP_ENABLED)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    scheduler = make_scheduler(optimizer, epochs * len(train_loader))

    history = []
    best_val_pr_auc = -np.inf
    best_state = None
    start_time = time.time()

    for epoch in range(1, epochs + 1):
        print(f"Epoch {epoch}/{epochs}")
        train_metrics, _, _ = run_epoch(model, train_loader, criterion, optimizer, f"train {epoch}", scheduler, scaler=scaler)
        val_metrics, _, _ = run_eval_epoch(model, val_loader, val_rc_loader, criterion, f"val {epoch}", bool(RC_TTA and EVAL_RC_TTA_EVERY_EPOCH))
        row = {"phase": "main", "epoch": epoch, "scheduler": SCHEDULER}
        row.update({f"train_{k}": v for k, v in train_metrics.items()})
        row.update({f"val_{k}": v for k, v in val_metrics.items()})
        history.append(row)
        print(row)
        if val_metrics["pr_auc"] > best_val_pr_auc:
            best_val_pr_auc = val_metrics["pr_auc"]
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            print("New best val_pr_auc:", f"{best_val_pr_auc:.4f}")

    if best_state is not None:
        model.load_state_dict(best_state)

    if hard_negative_epochs > 0:
        print("Hard-negative mining on train set...")
        _, train_probs, train_labels = run_eval_epoch(model, train_eval_loader, train_eval_rc_loader, criterion, "predict train", RC_TTA)
        pos_idx = train_indices[train_labels == 1]
        neg_idx = train_indices[train_labels == 0]
        neg_probs = train_probs[train_labels == 0]
        hard_count = min(len(neg_idx), int(np.ceil(len(pos_idx) * HARD_NEGATIVE_RATIO_TO_POS)))
        hard_neg_idx = neg_idx[np.argsort(-neg_probs)[:hard_count]]
        easy_pool = np.setdiff1d(neg_idx, hard_neg_idx, assume_unique=False)
        easy_count = min(len(easy_pool), int((len(pos_idx) + len(hard_neg_idx)) * EASY_NEGATIVE_RATIO))
        rng = np.random.default_rng(RANDOM_STATE)
        easy_neg_idx = rng.choice(easy_pool, size=easy_count, replace=False) if easy_count > 0 else np.asarray([], dtype=np.int64)
        hard_train_idx = np.concatenate([pos_idx, hard_neg_idx, easy_neg_idx])
        rng.shuffle(hard_train_idx)
        print("positive train rows:", f"{len(pos_idx):,}")
        print("top-score hard negative rows:", f"{len(hard_neg_idx):,}")
        print("sampled easy negative rows:", f"{len(easy_neg_idx):,}")
        print("fine-tune rows:", f"{len(hard_train_idx):,}")
        hard_sampler = make_weighted_sampler(hard_train_idx)
        hard_loader = make_loader(
            hard_train_idx,
            batch_size=batch_size,
            sampler=hard_sampler,
            shuffle=hard_sampler is None,
            random_reverse_complement=RC_AUGMENT,
        )
        optimizer = torch.optim.AdamW(model.parameters(), lr=HARD_NEGATIVE_LR, weight_decay=WEIGHT_DECAY)
        for epoch in range(1, hard_negative_epochs + 1):
            print(f"Hard-negative epoch {epoch}/{hard_negative_epochs}")
            train_metrics, _, _ = run_epoch(model, hard_loader, criterion, optimizer, f"hard train {epoch}", scaler=scaler)
            val_metrics, _, _ = run_eval_epoch(model, val_loader, val_rc_loader, criterion, f"hard val {epoch}", bool(RC_TTA and EVAL_RC_TTA_EVERY_EPOCH))
            row = {"phase": "hard_negative", "epoch": epoch, "scheduler": "none"}
            row.update({f"train_{k}": v for k, v in train_metrics.items()})
            row.update({f"val_{k}": v for k, v in val_metrics.items()})
            history.append(row)
            print(row)
            if val_metrics["pr_auc"] > best_val_pr_auc:
                best_val_pr_auc = val_metrics["pr_auc"]
                best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
                print("New best val_pr_auc:", f"{best_val_pr_auc:.4f}")

    if best_state is not None:
        model.load_state_dict(best_state)

    final_val_metrics, final_val_probs, final_val_labels = run_eval_epoch(model, val_loader, val_rc_loader, criterion, "final val", RC_TTA)
    test_metrics, test_probs, test_labels = run_eval_epoch(model, test_loader, test_rc_loader, criterion, "test", RC_TTA)
    val_thresholds = threshold_table(final_val_labels, final_val_probs)
    test_thresholds = threshold_table(test_labels, test_probs)
    best_val_f1 = val_thresholds.loc[val_thresholds["f1"].idxmax()].to_dict()
    best_val_f2 = val_thresholds.loc[val_thresholds["f2"].idxmax()].to_dict()
    val_recall_at_p80 = best_recall_at_precision(val_thresholds, 0.80)
    val_precision_at_r90 = best_precision_at_recall(val_thresholds, 0.90)
    test_at_best_val_f1 = metrics_at_threshold(test_labels, test_probs, best_val_f1["threshold"])
    test_at_best_val_f2 = metrics_at_threshold(test_labels, test_probs, best_val_f2["threshold"])
    test_at_val_p80 = None if val_recall_at_p80 is None else metrics_at_threshold(test_labels, test_probs, val_recall_at_p80["threshold"])
    test_at_val_r90 = None if val_precision_at_r90 is None else metrics_at_threshold(test_labels, test_probs, val_precision_at_r90["threshold"])

    metrics = {
        "model_name": run_name,
        "input_design": "sparse_alt_8ch_ref_onehot_plus_alt_at_center",
        "label_mode": LABEL_MODE,
        "selected_rows": int(len(active_idx)),
        "length": LENGTH,
        "sequence_channels": int(SEQ_CHANNELS),
        
        "model_variant": MODEL_VARIANT,
        "batch_size": int(batch_size),
        "num_workers": int(NUM_WORKERS),
        "use_amp": bool(AMP_ENABLED),
        "train_metrics_mode": TRAIN_METRICS_MODE,
        "eval_rc_tta_every_epoch": bool(EVAL_RC_TTA_EVERY_EPOCH),
        "final_rc_tta": bool(RC_TTA),
        
        
        "positional_encoding": POSITIONAL_ENCODING,
        "positional_dim": POSITIONAL_DIM,
        "split_audit": split_audit,
        "best_val_pr_auc": float(best_val_pr_auc),
        "final_val_metrics": final_val_metrics,
        "test_metrics": test_metrics,
        "best_val_f1_threshold": {k: float(v) for k, v in best_val_f1.items()},
        "best_val_f2_threshold": {k: float(v) for k, v in best_val_f2.items()},
        "val_recall_at_precision_80_threshold": None if val_recall_at_p80 is None else {k: float(v) for k, v in val_recall_at_p80.items()},
        "val_precision_at_recall_90_threshold": None if val_precision_at_r90 is None else {k: float(v) for k, v in val_precision_at_r90.items()},
        "test_at_best_val_f1_threshold": test_at_best_val_f1,
        "test_at_best_val_f2_threshold": test_at_best_val_f2,
        "test_at_val_recall_at_precision_80_threshold": test_at_val_p80,
        "test_at_val_precision_at_recall_90_threshold": test_at_val_r90,
        "train_rows": int(len(train_indices)),
        "val_rows": int(len(val_indices)),
        "test_rows": int(len(test_indices)),
        "imbalance_strategy": IMBALANCE_STRATEGY,
        "pos_weight": pos_weight_value,
        "runtime_minutes": float((time.time() - start_time) / 60.0),
    }

    predictions = metadata.iloc[test_indices].copy()
    predictions["y_true"] = test_labels
    predictions["pred_proba_pathogenic"] = test_probs
    predictions["pred_label_best_val_f1"] = (test_probs >= best_val_f1["threshold"]).astype(int)
    predictions["pred_label_best_val_f2"] = (test_probs >= best_val_f2["threshold"]).astype(int)

    if save_outputs:
        pd.DataFrame(history).to_parquet(paths["history"], index=False)
        predictions.to_parquet(paths["predictions"], index=False)
        test_thresholds.to_parquet(paths["thresholds"], index=False)
        val_thresholds.to_parquet(paths["val_thresholds"], index=False)
        np.savez_compressed(paths["split_indices"], train_idx=train_indices, val_idx=val_indices, test_idx=test_indices)
        paths["metrics"].write_text(json.dumps(metrics, indent=2), encoding="utf-8")
        torch.save({"model_state_dict": model.state_dict(), "config": metrics}, paths["model"])
        print("Saved metrics:", paths["metrics"])
        print("Saved model:", paths["model"])

    summary = {
        "model_name": run_name,
        "best_val_pr_auc": metrics["best_val_pr_auc"],
        "test_pr_auc": test_metrics["pr_auc"],
        "test_roc_auc": test_metrics["roc_auc"],
        "best_val_f1_threshold": best_val_f1["threshold"],
        "test_f1_at_best_val_f1_threshold": test_at_best_val_f1["f1_pathogenic"],
        "test_precision_at_best_val_f1_threshold": test_at_best_val_f1["precision_pathogenic"],
        "test_recall_at_best_val_f1_threshold": test_at_best_val_f1["recall_pathogenic"],
    }
    return model, metrics, predictions, pd.DataFrame(history), summary

## 15. Mini train

In [62]:
def stratified_subsample(indices, max_rows, seed):
    if len(indices) <= max_rows:
        return indices
    rng = np.random.default_rng(seed)
    selected = []
    for label in [0, 1]:
        label_idx = indices[y[indices] == label]
        n_label = int(round(max_rows * len(label_idx) / len(indices)))
        n_label = min(len(label_idx), max(1, n_label))
        selected.append(rng.choice(label_idx, size=n_label, replace=False))
    out = np.concatenate(selected)
    rng.shuffle(out)
    return out

if RUN_MINI_TRAIN:
    mini_train = stratified_subsample(train_idx, 2048, RANDOM_STATE)
    mini_val = stratified_subsample(val_idx, 512, RANDOM_STATE + 1)
    mini_test = stratified_subsample(test_idx, 512, RANDOM_STATE + 2)
    _, mini_metrics, _, mini_history, mini_summary = train_compact_model(
        mini_train, mini_val, mini_test,
        run_name=SAFE_NAME + "_debug",
        save_outputs=False,
        epochs=1,
        hard_negative_epochs=0,
        batch_size=64,
    )
    display(pd.Series(mini_summary))
    display(mini_history)
else:
    print("RUN_MINI_TRAIN=False")

RUN_MINI_TRAIN=False


## 16. Full train

In [ ]:
if RUN_FULL_TRAIN:
    model, metrics, predictions, history, summary = train_compact_model(train_idx, val_idx, test_idx, run_name=SAFE_NAME, save_outputs=True)
    display(pd.Series(summary))
    display(history)
else:
    print("RUN_FULL_TRAIN=False")

Epoch 1/5


/tmp/ipykernel_57/700434349.py:36: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=AMP_ENABLED)


train 1:   0%|          | 0/979 [00:00<?, ?it/s]

/tmp/ipykernel_57/304344902.py:33: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=AMP_ENABLED):


## 17. Load metrics

In [ ]:
rows = []
if paths["metrics"].exists():
    with open(paths["metrics"], "r", encoding="utf-8") as f:
        compact_metrics = json.load(f)
    rows.append({
        "model": compact_metrics.get("model_variant", MODEL_VARIANT),
        "model_name": compact_metrics["model_name"],
        "label_mode": compact_metrics.get("label_mode", LABEL_MODE),
        "test_pr_auc": compact_metrics["test_metrics"]["pr_auc"],
        "test_roc_auc": compact_metrics["test_metrics"]["roc_auc"],
        "best_val_pr_auc": compact_metrics["best_val_pr_auc"],
        "test_f1_at_best_val_f1": compact_metrics["test_at_best_val_f1_threshold"]["f1_pathogenic"],
        "precision_at_best_val_f1": compact_metrics["test_at_best_val_f1_threshold"]["precision_pathogenic"],
        "recall_at_best_val_f1": compact_metrics["test_at_best_val_f1_threshold"]["recall_pathogenic"],
    })
if rows:
    display(pd.DataFrame(rows))
else:
    print("No metrics found yet.")

## 18. Error analysis

Sort theo lỗi xác suất và dùng threshold chọn từ validation F1; không dùng threshold 0.5.

In [ ]:
if paths["predictions"].exists() and paths["metrics"].exists():
    predictions = pd.read_parquet(paths["predictions"])
    with open(paths["metrics"], "r", encoding="utf-8") as f:
        compact_metrics = json.load(f)
    threshold = compact_metrics["best_val_f1_threshold"]["threshold"]
    error_view = predictions.assign(
        pred_label_best_val_f1=lambda df: (df["pred_proba_pathogenic"] >= threshold).astype(int),
        error_score=lambda df: np.where(df["y_true"] == 1, 1.0 - df["pred_proba_pathogenic"], df["pred_proba_pathogenic"]),
    )
    display(error_view.sort_values("error_score", ascending=False)[[
        "GeneSymbol", "ClinicalSignificance", "ReferenceAlleleVCF", "AlternateAlleleVCF",
        "y_true", "pred_proba_pathogenic", "pred_label_best_val_f1", "error_score"
    ]].head(20))
else:
    print("No compact predictions/metrics found yet.")

In [ ]:
# 19. Attribution heatmap: model nhin quanh center hay vung xa?
# Chay cell nay sau khi full train xong. Cell load checkpoint + predictions, tinh saliency theo input REF one-hot.
from IPython.display import display, Markdown
import matplotlib.pyplot as plt

ATTRIBUTION_SAMPLES_PER_GROUP = 16

required_paths = [paths["model"], paths["predictions"], paths["metrics"], paths["split_indices"]]
missing = [str(p) for p in required_paths if not p.exists()]
if missing:
    raise FileNotFoundError("Thieu artifact de ve heatmap:\n" + "\n".join(missing))

with open(paths["metrics"], "r", encoding="utf-8") as f:
    compact_metrics = json.load(f)
threshold = compact_metrics["best_val_f1_threshold"]["threshold"]

pred_for_attr = pd.read_parquet(paths["predictions"])
split_file = np.load(paths["split_indices"])
pred_for_attr = pred_for_attr.copy()
pred_for_attr["dataset_index"] = split_file["test_idx"][:len(pred_for_attr)]
pred_for_attr["pred_label_best_val_f1"] = (pred_for_attr["pred_proba_pathogenic"] >= threshold).astype(int)
pred_for_attr["case_type"] = np.select(
    [
        (pred_for_attr["y_true"] == 1) & (pred_for_attr["pred_label_best_val_f1"] == 1),
        (pred_for_attr["y_true"] == 0) & (pred_for_attr["pred_label_best_val_f1"] == 0),
        (pred_for_attr["y_true"] == 0) & (pred_for_attr["pred_label_best_val_f1"] == 1),
        (pred_for_attr["y_true"] == 1) & (pred_for_attr["pred_label_best_val_f1"] == 0),
    ],
    ["true_positive", "true_negative", "false_positive", "false_negative"],
    default="other",
)

selected_parts = []
selected_parts.append(pred_for_attr[pred_for_attr["case_type"].eq("true_positive")].nlargest(ATTRIBUTION_SAMPLES_PER_GROUP, "pred_proba_pathogenic"))
selected_parts.append(pred_for_attr[pred_for_attr["case_type"].eq("true_negative")].nsmallest(ATTRIBUTION_SAMPLES_PER_GROUP, "pred_proba_pathogenic"))
selected_parts.append(pred_for_attr[pred_for_attr["case_type"].eq("false_positive")].nlargest(ATTRIBUTION_SAMPLES_PER_GROUP, "pred_proba_pathogenic"))
selected_parts.append(pred_for_attr[pred_for_attr["case_type"].eq("false_negative")].nsmallest(ATTRIBUTION_SAMPLES_PER_GROUP, "pred_proba_pathogenic"))
selected_attr = pd.concat(selected_parts, ignore_index=True).drop_duplicates("dataset_index")
if selected_attr.empty:
    selected_attr = pred_for_attr.sample(min(64, len(pred_for_attr)), random_state=RANDOM_STATE)

if "DilatedClinVarCNN" not in globals():
    raise RuntimeError("Hay chay cac cell dinh nghia model truoc cell heatmap nay.")

attr_model = make_sparse_alt_model()
try:
    checkpoint = torch.load(paths["model"], map_location=device, weights_only=False)
except TypeError:
    checkpoint = torch.load(paths["model"], map_location=device)
attr_model.load_state_dict(checkpoint["model_state_dict"] if isinstance(checkpoint, dict) and "model_state_dict" in checkpoint else checkpoint)
attr_model.eval()

attr_ds = SparseAlt8chDataset(selected_attr["dataset_index"].to_numpy(dtype=np.int64))
seq_list, label_list = [], []
for item in range(len(attr_ds)):
    seq_item, label_item = attr_ds[item]
    seq_list.append(seq_item)
    label_list.append(label_item)

seq_batch = torch.stack(seq_list).to(device)
seq_batch.requires_grad_(True)

attr_model.zero_grad(set_to_none=True)
logits = attr_model(seq_batch)
# Saliency theo class pathogenic: gradient cua tong logit pathogenic doi voi input sequence.
logits.sum().backward()

seq_np = seq_batch.detach().cpu().numpy()
grad_np = seq_batch.grad.detach().cpu().numpy()
# Chi lay 4 kenh base REF. Neu sau nay bat positional, positional khong nam trong heatmap base nay.
base_attr = np.abs(grad_np[:, :4, :] * seq_np[:, :4, :])
base_heatmap = base_attr.mean(axis=0)
position_attr = base_heatmap.sum(axis=0)
position_attr_norm = position_attr / (position_attr.max() + 1e-12)
base_heatmap_norm = base_heatmap / (base_heatmap.max() + 1e-12)
relative_position = np.arange(LENGTH) - FLANK

def attribution_share(radius):
    mask = np.abs(relative_position) <= radius
    return float(position_attr[mask].sum() / (position_attr.sum() + 1e-12))

share_table = pd.DataFrame([
    {"region": "+/-10 bp quanh center", "attribution_share": attribution_share(10)},
    {"region": "+/-50 bp quanh center", "attribution_share": attribution_share(50)},
    {"region": "+/-150 bp quanh center", "attribution_share": attribution_share(150)},
    {"region": "ngoai +/-150 bp", "attribution_share": 1.0 - attribution_share(150)},
])

fig, axes = plt.subplots(2, 1, figsize=(15, 6), gridspec_kw={"height_ratios": [1, 1.3]}, constrained_layout=True)
im = axes[0].imshow(
    base_heatmap_norm,
    aspect="auto",
    cmap="magma",
    extent=[relative_position[0], relative_position[-1], 3.5, -0.5],
)
axes[0].set_yticks(range(4))
axes[0].set_yticklabels(["A", "C", "G", "T"])
axes[0].axvline(0, color="cyan", linestyle="--", linewidth=1)
axes[0].set_title("Average gradient x input saliency theo base REF")
axes[0].set_xlabel("Vi tri tuong doi so voi SNV center (bp)")
axes[0].set_ylabel("Base channel")
fig.colorbar(im, ax=axes[0], fraction=0.025, pad=0.01, label="normalized saliency")

axes[1].plot(relative_position, position_attr_norm, color="#1f77b4", linewidth=1.2)
axes[1].axvline(0, color="black", linestyle="--", linewidth=1)
axes[1].axvspan(-10, 10, color="#ffcc66", alpha=0.25, label="+/-10 bp")
axes[1].axvspan(-50, 50, color="#66c2a5", alpha=0.12, label="+/-50 bp")
axes[1].set_title("Saliency tong theo vi tri")
axes[1].set_xlabel("Vi tri tuong doi so voi SNV center (bp)")
axes[1].set_ylabel("normalized saliency")
axes[1].legend(loc="upper right")
plt.show()

display(Markdown(
    "**Cach doc heatmap:** mau nong gan `0 bp` nghia la model tap trung vao motif quanh dot bien. "
    "Neu duong saliency con cao o xa center hoac `ngoai +/-150 bp` lon, model dang dung them tin hieu vung xa trong 601 bp. "
    "Day la saliency gradient x input, nen chi la bang chung goi y, khong phai chung minh nhan qua sinh hoc."
))
display(share_table)
display(selected_attr[["case_type", "GeneSymbol", "ClinicalSignificance", "ReferenceAlleleVCF", "AlternateAlleleVCF", "y_true", "pred_proba_pathogenic"]].head(12))

In [ ]:
# 20. Error analysis: cac case model tu tin sai nhat
# Tap trung vao false positive ~99% nhung label benign, va false negative ~0% nhung label pathogenic.
from IPython.display import display, Markdown

required_paths = [paths["predictions"], paths["metrics"], paths["split_indices"]]
missing = [str(p) for p in required_paths if not p.exists()]
if missing:
    raise FileNotFoundError("Thieu artifact de phan tich loi:\n" + "\n".join(missing))

with open(paths["metrics"], "r", encoding="utf-8") as f:
    compact_metrics = json.load(f)
threshold = compact_metrics["best_val_f1_threshold"]["threshold"]

err = pd.read_parquet(paths["predictions"]).copy()
split_file = np.load(paths["split_indices"])
err["dataset_index"] = split_file["test_idx"][:len(err)]
err["mutation"] = err["ReferenceAlleleVCF"].astype(str) + ">" + err["AlternateAlleleVCF"].astype(str)
err["pred_label_best_val_f1"] = (err["pred_proba_pathogenic"] >= threshold).astype(int)
err["is_wrong"] = err["pred_label_best_val_f1"] != err["y_true"]
err["error_type"] = np.select(
    [
        (err["y_true"] == 0) & (err["pred_label_best_val_f1"] == 1),
        (err["y_true"] == 1) & (err["pred_label_best_val_f1"] == 0),
    ],
    ["false_positive_benign_called_pathogenic", "false_negative_pathogenic_called_benign"],
    default="correct",
)
err["confidence_wrong"] = np.where(err["y_true"] == 0, err["pred_proba_pathogenic"], 1.0 - err["pred_proba_pathogenic"])
wrong = err[err["is_wrong"]].sort_values("confidence_wrong", ascending=False).copy()

false_positive_99 = err[(err["y_true"] == 0) & (err["pred_proba_pathogenic"] >= 0.99)].sort_values("pred_proba_pathogenic", ascending=False)
false_negative_01 = err[(err["y_true"] == 1) & (err["pred_proba_pathogenic"] <= 0.01)].sort_values("pred_proba_pathogenic", ascending=True)

cols = [
    "GeneSymbol", "ClinicalSignificance", "ReviewStatus", "Chromosome", "PositionVCF",
    "mutation", "y_true", "pred_proba_pathogenic", "pred_label_best_val_f1", "confidence_wrong",
]

display(Markdown(f"""
**Nguong dang dung:** best validation F1 threshold = `{threshold:.4f}`.  
**Tong loi o test:** `{len(wrong):,}` / `{len(err):,}`.  
**Benign/Likely benign nhung model cho >=99% pathogenic:** `{len(false_positive_99):,}` case.  
**Pathogenic/Likely pathogenic nhung model cho <=1% pathogenic:** `{len(false_negative_01):,}` case.
"""))

display(Markdown("### Top case tu tin sai nhat"))
display(wrong[cols].head(25))

display(Markdown("### Benign/Likely benign bi goi pathogenic >=99%"))
display(false_positive_99[cols].head(20))

display(Markdown("### Pathogenic/Likely pathogenic bi goi benign <=1%"))
display(false_negative_01[cols].head(20))

summary_by_type = wrong.groupby("error_type").agg(
    n=("error_type", "size"),
    median_pred_proba=("pred_proba_pathogenic", "median"),
    median_confidence_wrong=("confidence_wrong", "median"),
).reset_index().sort_values("n", ascending=False)

mutation_summary = wrong.groupby(["error_type", "mutation"]).agg(
    n=("mutation", "size"),
    median_pred_proba=("pred_proba_pathogenic", "median"),
).reset_index().sort_values(["error_type", "n"], ascending=[True, False])

review_summary = wrong.groupby(["error_type", "ReviewStatus"]).agg(
    n=("ReviewStatus", "size"),
    median_confidence_wrong=("confidence_wrong", "median"),
).reset_index().sort_values(["error_type", "n"], ascending=[True, False])

gene_summary = wrong.groupby(["error_type", "GeneSymbol"]).agg(
    n=("GeneSymbol", "size"),
    max_confidence_wrong=("confidence_wrong", "max"),
    median_pred_proba=("pred_proba_pathogenic", "median"),
).reset_index().sort_values(["error_type", "n", "max_confidence_wrong"], ascending=[True, False, False])

display(Markdown("### Loi gom theo loai"))
display(summary_by_type)

display(Markdown("### Mutation type hay sai nhat"))
display(mutation_summary.groupby("error_type").head(8))

display(Markdown("### Review status trong cac case sai"))
display(review_summary.groupby("error_type").head(8))

display(Markdown("### Gene xuat hien nhieu trong cac case sai"))
display(gene_summary.groupby("error_type").head(12))

display(Markdown(
    "**Cach giai thich cac case 99% nhung label 0:** day la false positive tu tin cao. "
    "Thuong can xem `ReviewStatus`, gene, va mutation type. Neu review status yeu hoac nhan ClinVar dang tranh cai/gop nhan, day co the la label noise. "
    "Neu review status manh, kha nang cao model dang hoc motif/locus/pathogenic-like context nhung thieu annotation ngoai sequence nhu protein consequence, allele frequency, conservation, gene constraint.\n\n"
    "**Cach giai thich cac case gan 0% nhung label 1:** day la false negative tu tin cao. "
    "Nhung bien the nay co the gay benh boi co che khong the hien ro trong 601 bp raw DNA context, vi du tac dong protein, splice annotation xa hon, inheritance/gene constraint, hoac thong tin ClinVar/meta-data ma model sequence khong thay."
))

In [ ]:
# 21. ReviewStatus cross-check cho cac loi tu tin cao
# Dinh nghia "tu tin cao nhung sai": false positive co proba >= 0.99 hoac false negative co proba <= 0.01.
from IPython.display import display, Markdown

HIGH_CONF_WRONG_THRESHOLD = 0.99
required_paths = [paths["predictions"], paths["metrics"]]
missing = [str(p) for p in required_paths if not p.exists()]
if missing:
    raise FileNotFoundError("Thieu artifact de doi chieu ReviewStatus:\n" + "\n".join(missing))

with open(paths["metrics"], "r", encoding="utf-8") as f:
    compact_metrics = json.load(f)
threshold = compact_metrics["best_val_f1_threshold"]["threshold"]

review_df = pd.read_parquet(paths["predictions"]).copy()
review_df["pred_label_best_val_f1"] = (review_df["pred_proba_pathogenic"] >= threshold).astype(int)
review_df["is_wrong"] = review_df["pred_label_best_val_f1"].ne(review_df["y_true"])
review_df["error_type"] = np.select(
    [
        (review_df["y_true"] == 0) & (review_df["pred_label_best_val_f1"] == 1),
        (review_df["y_true"] == 1) & (review_df["pred_label_best_val_f1"] == 0),
    ],
    ["false_positive_benign_called_pathogenic", "false_negative_pathogenic_called_benign"],
    default="correct",
)
review_df["confidence_wrong"] = np.where(
    review_df["y_true"] == 0,
    review_df["pred_proba_pathogenic"],
    1.0 - review_df["pred_proba_pathogenic"],
)
review_df["high_conf_wrong"] = review_df["is_wrong"] & review_df["confidence_wrong"].ge(HIGH_CONF_WRONG_THRESHOLD)
review_df["mutation"] = review_df["ReferenceAlleleVCF"].astype(str) + ">" + review_df["AlternateAlleleVCF"].astype(str)

wrong_review = review_df[review_df["is_wrong"]].copy()
high_review = review_df[review_df["high_conf_wrong"]].copy()

high_by_review = high_review.groupby(["error_type", "ReviewStatus"]).agg(
    n=("ReviewStatus", "size"),
    median_confidence_wrong=("confidence_wrong", "median"),
    min_confidence_wrong=("confidence_wrong", "min"),
    max_confidence_wrong=("confidence_wrong", "max"),
).reset_index().sort_values(["error_type", "n"], ascending=[True, False])

all_wrong_by_review = wrong_review.groupby(["error_type", "ReviewStatus"]).agg(
    n=("ReviewStatus", "size"),
    median_confidence_wrong=("confidence_wrong", "median"),
).reset_index().sort_values(["error_type", "n"], ascending=[True, False])

review_denominator = review_df.groupby("ReviewStatus").agg(
    total=("ReviewStatus", "size"),
    pathogenic_rows=("y_true", "sum"),
    wrong=("is_wrong", "sum"),
    high_conf_wrong=("high_conf_wrong", "sum"),
).reset_index()
review_denominator["benign_rows"] = review_denominator["total"] - review_denominator["pathogenic_rows"]
review_denominator["wrong_rate"] = review_denominator["wrong"] / review_denominator["total"]
review_denominator["high_conf_wrong_rate"] = review_denominator["high_conf_wrong"] / review_denominator["total"]
review_denominator = review_denominator.sort_values("high_conf_wrong", ascending=False)

high_cols = [
    "GeneSymbol", "ClinicalSignificance", "ReviewStatus", "Chromosome", "PositionVCF",
    "mutation", "y_true", "pred_proba_pathogenic", "confidence_wrong",
]

display(Markdown(f"""
**Cross-check ReviewStatus cho lỗi tự tin cao**  
Ngưỡng quyết định đang dùng: best-val-F1 threshold = `{threshold:.4f}`.  
Lỗi tự tin cao được định nghĩa bằng `confidence_wrong >= {HIGH_CONF_WRONG_THRESHOLD}`.

- Tổng test rows: `{len(review_df):,}`
- Tổng lỗi theo threshold validation: `{len(wrong_review):,}`
- Lỗi tự tin cao: `{len(high_review):,}`
- False positive tự tin cao: `{int((high_review['error_type'] == 'false_positive_benign_called_pathogenic').sum()):,}`
- False negative tự tin cao: `{int((high_review['error_type'] == 'false_negative_pathogenic_called_benign').sum()):,}`
"""))

display(Markdown("### Lỗi tự tin cao theo ReviewStatus"))
display(high_by_review)

display(Markdown("### Tất cả lỗi theo ReviewStatus"))
display(all_wrong_by_review)

display(Markdown("### Mẫu số theo ReviewStatus và tỉ lệ lỗi"))
display(review_denominator)

display(Markdown("### Các case tự tin cao sai nhất"))
display(high_review.sort_values("confidence_wrong", ascending=False)[high_cols].head(30))

display(Markdown(
    "**Nhận xét cần ghi vào báo cáo:** nếu lỗi tự tin cao chủ yếu nằm ở `criteria provided, single submitter`, "
    "có thể nghi ngờ một phần là label noise hoặc bằng chứng ClinVar yếu hơn. Nhưng nếu cũng xuất hiện ở "
    "`multiple submitters, no conflicts`, `expert panel`, hoặc `practice guideline`, thì không nên đổ hết cho nhãn; "
    "khi đó model sequence đang thiếu thông tin ngoài 601 bp như consequence protein/splicing, allele frequency, conservation, gene constraint, "
    "hoặc cơ chế bệnh không nằm trong raw local DNA context."
))

## 22. Calibration plot + Platt scaling

Kiểm tra xem probability model output có khớp tỉ lệ pathogenic thực tế không.

- **Calibration plot**: chia predictions thành 10 bins, so sánh predicted probability vs actual positive rate.
- **Platt scaling**: fit logistic regression 2 tham số (A, B) trên **validation set**, apply lên test → chỉnh probability scale.
- **So sánh**: vẽ calibration trước/sau Platt, bảng metrics trước/sau (ROC-AUC, PR-AUC không đổi; ECE giảm).

In [ ]:
# 22. Calibration plot + Platt scaling
from IPython.display import display, Markdown
import matplotlib.pyplot as plt
from sklearn.calibration import calibration_curve
from sklearn.linear_model import LogisticRegression
from scipy.special import expit as sigmoid, logit as scipy_logit

required = [paths["predictions"], paths["metrics"], paths["split_indices"]]
missing = [str(p) for p in required if not p.exists()]
if missing:
    raise FileNotFoundError("Thieu artifact:\n" + "\n".join(missing))

with open(paths["metrics"], "r", encoding="utf-8") as f:
    cal_metrics = json.load(f)

test_pred = pd.read_parquet(paths["predictions"]).copy()
y_test = test_pred["y_true"].to_numpy(dtype=int)
prob_test_raw = test_pred["pred_proba_pathogenic"].to_numpy(dtype=np.float64)

# --- Load validation predictions ---
# Re-run model on validation set to get val probabilities for Platt fitting.
# If split_indices exist and model is loaded, predict on val; otherwise use a simple fallback.
split_data = np.load(paths["split_indices"])
val_idx = split_data["val_idx"]

if "model" not in dir() or model is None:
    # Try to load model
    _model = make_sparse_alt_model().to(device)
    try:
        _ckpt = torch.load(paths["model"], map_location=device, weights_only=False)
    except TypeError:
        _ckpt = torch.load(paths["model"], map_location=device)
    _model.load_state_dict(_ckpt["model_state_dict"] if isinstance(_ckpt, dict) and "model_state_dict" in _ckpt else _ckpt)
    _model.eval()
else:
    _model = model

val_loader = make_loader(val_idx, batch_size=BATCH_SIZE)
val_rc_loader = make_loader(val_idx, batch_size=BATCH_SIZE, reverse_complement=True) if RC_TTA else None
_criterion = nn.BCEWithLogitsLoss()
_, prob_val, y_val = run_eval_epoch(_model, val_loader, val_rc_loader, _criterion, "val for calibration", RC_TTA)

print(f"Validation set: {len(y_val):,} rows, pathogenic rate: {y_val.mean():.4f}")
print(f"Test set: {len(y_test):,} rows, pathogenic rate: {y_test.mean():.4f}")

# --- Platt scaling: fit on validation, apply on test ---
prob_val_clip = np.clip(prob_val, 1e-7, 1 - 1e-7)
logits_val = scipy_logit(prob_val_clip)

platt = LogisticRegression(C=1e10, solver="lbfgs", max_iter=1000)
platt.fit(logits_val.reshape(-1, 1), y_val)

prob_test_clip = np.clip(prob_test_raw, 1e-7, 1 - 1e-7)
logits_test = scipy_logit(prob_test_clip)
prob_test_calibrated = platt.predict_proba(logits_test.reshape(-1, 1))[:, 1]

print(f"\nPlatt coefficients: A={platt.coef_[0][0]:.4f}, B={platt.intercept_[0]:.4f}")
print(f"Interpretation: calibrated = sigmoid({platt.coef_[0][0]:.4f} * logit + {platt.intercept_[0]:.4f})")

# --- Calibration curves ---
prob_true_raw, prob_pred_raw = calibration_curve(y_test, prob_test_raw, n_bins=10, strategy="uniform")
prob_true_cal, prob_pred_cal = calibration_curve(y_test, prob_test_calibrated, n_bins=10, strategy="uniform")

# --- ECE (Expected Calibration Error) ---
def compute_ece(y_true, y_prob, n_bins=10):
    bin_edges = np.linspace(0, 1, n_bins + 1)
    ece = 0.0
    for lo, hi in zip(bin_edges[:-1], bin_edges[1:]):
        mask = (y_prob >= lo) & (y_prob < hi)
        if mask.sum() == 0:
            continue
        bin_acc = y_true[mask].mean()
        bin_conf = y_prob[mask].mean()
        ece += mask.sum() * abs(bin_acc - bin_conf)
    return ece / len(y_true)

ece_raw = compute_ece(y_test, prob_test_raw)
ece_cal = compute_ece(y_test, prob_test_calibrated)

# --- Plot ---
fig, axes = plt.subplots(1, 3, figsize=(18, 5.5), constrained_layout=True)

# 1) Calibration plot: before vs after
ax = axes[0]
ax.plot([0, 1], [0, 1], "--", color="#999999", linewidth=1, label="Perfect")
ax.plot(prob_pred_raw, prob_true_raw, "o-", color="#e74c3c", linewidth=2, markersize=7, label=f"Raw (ECE={ece_raw:.4f})")
ax.plot(prob_pred_cal, prob_true_cal, "s-", color="#2ecc71", linewidth=2, markersize=7, label=f"Platt (ECE={ece_cal:.4f})")
ax.set_xlabel("Predicted probability", fontsize=12)
ax.set_ylabel("Actual pathogenic rate", fontsize=12)
ax.set_title("Calibration: Raw vs Platt scaling", fontsize=13, fontweight="bold")
ax.legend(fontsize=10)
ax.set_xlim(-0.02, 1.02)
ax.set_ylim(-0.02, 1.02)
ax.grid(True, alpha=0.3)

# 2) Gap per bin
ax = axes[1]
bin_centers_raw = prob_pred_raw
gap_raw = prob_pred_raw - prob_true_raw
gap_cal = prob_pred_cal - prob_true_cal
bar_width = 0.035
ax.bar(bin_centers_raw - bar_width, gap_raw, width=bar_width * 1.8, color="#e74c3c", alpha=0.7, label="Raw gap")
ax.bar(prob_pred_cal + bar_width, gap_cal, width=bar_width * 1.8, color="#2ecc71", alpha=0.7, label="Platt gap")
ax.axhline(0, color="black", linewidth=0.8)
ax.set_xlabel("Predicted probability", fontsize=12)
ax.set_ylabel("Gap (predicted - actual)", fontsize=12)
ax.set_title("Overconfidence gap per bin", fontsize=13, fontweight="bold")
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

# 3) Probability distribution shift
ax = axes[2]
bins_hist = np.linspace(0, 1, 51)
ax.hist(prob_test_raw[y_test == 0], bins=bins_hist, alpha=0.4, color="#3498db", label="Benign (raw)", density=True)
ax.hist(prob_test_raw[y_test == 1], bins=bins_hist, alpha=0.4, color="#e74c3c", label="Pathogenic (raw)", density=True)
ax.hist(prob_test_calibrated[y_test == 0], bins=bins_hist, alpha=0.3, color="#3498db", histtype="step", linewidth=2, label="Benign (Platt)", density=True, linestyle="--")
ax.hist(prob_test_calibrated[y_test == 1], bins=bins_hist, alpha=0.3, color="#e74c3c", histtype="step", linewidth=2, label="Pathogenic (Platt)", density=True, linestyle="--")
ax.set_xlabel("Predicted probability", fontsize=12)
ax.set_ylabel("Density", fontsize=12)
ax.set_title("Probability distribution: Raw vs Platt", fontsize=13, fontweight="bold")
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

plt.savefig(PROCESSED_DIR / f"{SAFE_NAME}_calibration_plot.png", dpi=150, bbox_inches="tight")
plt.show()

# --- Summary table ---
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss

threshold = cal_metrics["best_val_f1_threshold"]["threshold"]

def compute_summary(y_true, y_prob, label):
    pred_label = (y_prob >= threshold).astype(int)
    tp = ((pred_label == 1) & (y_true == 1)).sum()
    fp = ((pred_label == 1) & (y_true == 0)).sum()
    fn = ((pred_label == 0) & (y_true == 1)).sum()
    prec = tp / max(tp + fp, 1)
    rec = tp / max(tp + fn, 1)
    f1 = 2 * prec * rec / max(prec + rec, 1e-12)
    return {
        "version": label,
        "ROC-AUC": roc_auc_score(y_true, y_prob),
        "PR-AUC": average_precision_score(y_true, y_prob),
        "Brier score": brier_score_loss(y_true, y_prob),
        "ECE": compute_ece(y_true, y_prob),
        f"Precision @{threshold:.3f}": prec,
        f"Recall @{threshold:.3f}": rec,
        f"F1 @{threshold:.3f}": f1,
    }

comparison = pd.DataFrame([
    compute_summary(y_test, prob_test_raw, "Raw"),
    compute_summary(y_test, prob_test_calibrated, "Platt"),
]).set_index("version")

display(Markdown("### So sánh trước/sau Platt scaling"))
display(Markdown(
    "ROC-AUC và PR-AUC **không đổi** vì Platt giữ nguyên ranking. "
    "Brier score và ECE **giảm** vì probability sát thực tế hơn. "
    "Precision/Recall/F1 tại threshold cũ có thể thay đổi vì probability scale dịch."
))
display(comparison.T)

# --- Binned calibration table ---
display(Markdown("### Bảng calibration chi tiết"))
cal_table_rows = []
for lo in np.arange(0, 1, 0.1):
    hi = lo + 0.1
    mask_raw = (prob_test_raw >= lo) & (prob_test_raw < hi)
    mask_cal = (prob_test_calibrated >= lo) & (prob_test_calibrated < hi)
    row = {"bin": f"{lo:.1f}-{hi:.1f}"}
    if mask_raw.sum() > 0:
        row["n_raw"] = int(mask_raw.sum())
        row["actual_raw"] = f"{y_test[mask_raw].mean():.4f}"
        row["pred_raw"] = f"{prob_test_raw[mask_raw].mean():.4f}"
        row["gap_raw"] = f"{prob_test_raw[mask_raw].mean() - y_test[mask_raw].mean():+.4f}"
    if mask_cal.sum() > 0:
        row["n_platt"] = int(mask_cal.sum())
        row["actual_platt"] = f"{y_test[mask_cal].mean():.4f}"
        row["pred_platt"] = f"{prob_test_calibrated[mask_cal].mean():.4f}"
        row["gap_platt"] = f"{prob_test_calibrated[mask_cal].mean() - y_test[mask_cal].mean():+.4f}"
    cal_table_rows.append(row)
display(pd.DataFrame(cal_table_rows).fillna("-"))
